# Week 17, Day 3: Fisher information and the limits of precision

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w017/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The MLE reaches the bound

For a rate of 3 per trial, simulate 20,000 experiments with n = 3 and n = 40 trials. Compute the variance of the MLE for n = 40 (`var_mle`) and compare with the Cramér-Rao bound λ/n. For both n, compute how often the Wald interval mean ± 1.96 sqrt(mean / n) contains 3, storing the fractions in `coverage[n]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2014)
lam = 3.0
coverage = {}
for n in (3, 40):
    mle = rng.poisson(lam, size=(20000, n)).mean(axis=1)
    half = ...
    coverage[n] = ...
    if n == 40:
        var_mle = mle.var()
print(f"variance of the MLE with 40 trials {var_mle:.4f}, Cramer-Rao bound {lam / 40:.4f}")
print("Wald coverage:", {n: round(c, 3) for n, c in coverage.items()})

In [ ]:
import numpy as np
def test_var():
    "with 40 trials, the variance of the MLE across experiments is within 5% of lambda / n"
    assert abs(var_mle - 3 / 40) / (3 / 40) < 0.05

def test_coverage():
    "Wald intervals cover the truth about 95% of the time with 40 trials (0.93 to 0.97) but less well with 3 trials"
    assert 0.93 < coverage[40] < 0.97 and coverage[3] < coverage[40]

test_var()
test_coverage()
print("All checks passed")

### Exercise: Fisher information and a real decoder

For 32 neurons with Gaussian tuning curves (width 20°, peak 20 spikes, baseline 1), write `pop_fi(theta, prefs)`. Decode 3,000 trials at θ = 60° with a maximum likelihood grid decoder and compare the decoder's variance `dec_var` with 1/I. Then compute `fi_ratio`, the information at 60° with 64 neurons over that with 32.

In [ ]:
def circ(d): return (d + 90) % 180 - 90
def tuning(theta, prefs): return 1 + 20 * np.exp(-circ(np.subtract.outer(theta, prefs)) ** 2 / (2 * 20 ** 2))
def fprime(theta, prefs, h=1e-4): return (tuning(theta + h, prefs) - tuning(theta - h, prefs)) / (2 * h)
prefs = np.arange(32) * 180 / 32

def pop_fi(theta, prefs):
    return ...                                   # sum over neurons of f'^2 / f, one value per theta

grid = np.arange(40, 80, 0.05)
F = tuning(grid, prefs)
est = []
for _ in range(3000):
    k = rng.poisson(tuning(60.0, prefs))
    est.append(grid[np.argmax((k * np.log(F) - F).sum(axis=1))])
dec_var = np.var(est)
fi_ratio = ...
print(f"Fisher information at 60 deg: {pop_fi(np.array([60.0]), prefs)[0]:.3f} per deg^2;  bound on variance {1 / pop_fi(np.array([60.0]), prefs)[0]:.3f};  ML decoder variance {dec_var:.3f}")
print(f"information ratio, 64 against 32 neurons: {fi_ratio:.3f}")

In [ ]:
import numpy as np
def test_fi():
    "pop_fi computes the sum of f'^2 / f"
    assert abs(pop_fi(np.array([60.0]), prefs)[0] - np.sum(fprime(60.0, prefs) ** 2 / tuning(60.0, prefs))) < 1e-9

def test_decoder():
    "the ML decoder's variance at 60 degrees is within 15% of the Cramer-Rao bound 1 / I"
    assert abs(dec_var - 1 / pop_fi(np.array([60.0]), prefs)[0]) * pop_fi(np.array([60.0]), prefs)[0] < 0.15

def test_doubling():
    "doubling the number of neurons roughly doubles the information"
    assert 1.9 < fi_ratio < 2.1

test_fi()
test_decoder()
test_doubling()
print("All checks passed")

### Exercise: Information saturates

For N = 10, 30, 100, 300 and 1000 neurons tiling orientation, compute the linear Fisher information at 60° with independent Poisson-like noise (Σ0 = diag(f)) as `info_indep`, and with added differential correlations ε f′f′ᵀ (ε = 0.2) as `info_corr`, using np.linalg.solve.

In [ ]:
eps = 0.2
Ns = [10, 30, 100, 300, 1000]
info_indep, info_corr = [], []
for N in Ns:
    p = np.arange(N) * 180 / N
    fpv = fprime(60.0, p)
    S0 = np.diag(tuning(60.0, p))
    info_indep.append(...)
    S = S0 + eps * np.outer(fpv, fpv)
    info_corr.append(...)
info_indep, info_corr = np.array(info_indep), np.array(info_corr)
print("independent:", info_indep.round(2)); print("differential correlations:", info_corr.round(2), " limit 1/eps =", 1 / eps)

In [ ]:
import numpy as np
def test_formula():
    "the computed information matches I0 / (1 + eps I0) for every N"
    assert np.allclose(info_corr, info_indep / (1 + eps * info_indep), rtol=1e-6)

def test_saturates():
    "independent information grows about 100-fold from N = 10 to 1000, but the correlated information stays below 1 / eps"
    assert info_indep[-1] / info_indep[0] > 80 and np.all(info_corr < 1 / eps)

test_formula()
test_saturates()
print("All checks passed")